In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2015_2016"
    / "Labs"
)

xpt_file = DATA_DIR / "GHB_I.xpt"
csv_file = DATA_DIR / "GHB_I_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve the completely untouched pandas-decoded
# source for fidelity comparison.
#
source_df = df.copy()


print("=" * 100)
print("NHANES 2015-2016 GHB_I XPT -> CSV")
print("=" * 100)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "LBXGH"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "GHB_I column names/order differ.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 6744:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 2:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 6,744 rows x 2 columns."
)


# ============================================================
# 4. DEFINE FIELD TYPES
# ============================================================

#
# GHB_I is a mixed ID / continuous laboratory file.
#
#
# SEQN
#     Respondent sequence number.
#     Whole-number identifier.
#
#
# LBXGH
#     Glycohemoglobin / HbA1c (%).
#     Genuine continuous decimal laboratory measurement.
#
#
# Therefore:
#
#     SEQN  -> Int64
#
#     LBXGH -> KEEP FLOAT AS IS
#
#
# IMPORTANT:
#
# DO NOT:
#
#     round LBXGH
#
# DO NOT:
#
#     convert LBXGH to Int64
#
# DO NOT:
#
#     globally remove ".0"
#
# DO NOT:
#
#     use float_format=
#
# when writing the CSV.
#

DECIMAL_COLUMNS = [
    "LBXGH"
]


INTEGER_COLUMNS = [
    "SEQN"
]


if len(DECIMAL_COLUMNS) != 1:

    raise ValueError(
        "Expected exactly 1 genuine decimal field."
    )


if len(INTEGER_COLUMNS) != 1:

    raise ValueError(
        "Expected exactly 1 integer/ID field."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/ID fields:",
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas / SAS XPORT numeric-zero artifact:
#
#     5.397605346934028e-79
#
#
# The attached GHB_I contains NONE.
#
# Therefore NO tiny -> zero correction is needed.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected both GHB_I fields "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Ordinary numeric zeros:",
    f"{true_zero_total:,}"
)


if tiny_total > 0:

    print(
        "\nUnexpected tiny-value pattern:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT values "
        "found in GHB_I."
    )


if true_zero_total > 0:

    print(
        "\nUnexpected ordinary zeros:"
    )

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected numeric zero values "
        "found in GHB_I."
    )


print(
    "PASS: GHB_I contains 0 tiny artifacts "
    "and 0 ordinary numeric zeros."
)


# ============================================================
# 6. NO ZERO RESTORATION REQUIRED
# ============================================================

#
# This file requires NO numeric zero repair.
#
# Therefore the original XPT values are directly
# the fidelity reference.
#

expected_df = source_df.copy()


print(
    "\nPASS: No tiny-to-zero correction required."
)


# ============================================================
# 7. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# This check occurs BEFORE any Int64 conversion.
#
# np.round() below is used ONLY to determine whether
# a value has a fractional component.
#
# It does NOT alter the dataframe.
#

print(
    "\n--- GENUINE DECIMAL DETECTION ---"
)


detected_decimal_columns = []

fractional_counts = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_counts[col] = 0
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    fractional_counts[col] = fractional_count


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        detected_decimal_columns.append(
            col
        )


if detected_decimal_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "STOP: Decimal-variable pattern differs.\n\n"
        f"Expected:\n{DECIMAL_COLUMNS}\n\n"
        f"Actual:\n{detected_decimal_columns}"
    )


print(
    "\nPASS: LBXGH is the only genuine "
    "decimal field."
)


# ============================================================
# 8. LBXGH DECIMAL DETAILS
# ============================================================

lbxgh = df["LBXGH"]


lbxgh_nonmissing = int(
    lbxgh.notna()
    .sum()
)


lbxgh_missing = int(
    lbxgh.isna()
    .sum()
)


lbxgh_values = (
    lbxgh.dropna()
    .to_numpy(
        dtype=float
    )
)


lbxgh_fractional = int(
    (
        ~np.isclose(
            lbxgh_values,
            np.round(lbxgh_values),
            rtol=0,
            atol=1e-12
        )
    )
    .sum()
)


lbxgh_whole_looking = (
    lbxgh_nonmissing
    -
    lbxgh_fractional
)


lbxgh_min = float(
    lbxgh.min()
)


lbxgh_max = float(
    lbxgh.max()
)


print("\n--- LBXGH DECIMAL DETAILS ---")

print(
    "Nonmissing:",
    f"{lbxgh_nonmissing:,}"
)

print(
    "Missing:",
    f"{lbxgh_missing:,}"
)

print(
    "Minimum:",
    repr(lbxgh_min)
)

print(
    "Maximum:",
    repr(lbxgh_max)
)

print(
    "Fractional observations:",
    f"{lbxgh_fractional:,}"
)

print(
    "Whole-looking observations:",
    f"{lbxgh_whole_looking:,}"
)


if lbxgh_nonmissing != 6326:

    raise ValueError(
        "Unexpected LBXGH nonmissing count."
    )


if lbxgh_missing != 418:

    raise ValueError(
        "Unexpected LBXGH missing count."
    )


if lbxgh_min != 3.8:

    raise ValueError(
        "Unexpected LBXGH minimum."
    )


if lbxgh_max != 17.0:

    raise ValueError(
        "Unexpected LBXGH maximum."
    )


if lbxgh_fractional != 5714:

    raise ValueError(
        "Unexpected fractional LBXGH count."
    )


if lbxgh_whole_looking != 612:

    raise ValueError(
        "Unexpected whole-looking LBXGH count."
    )


print(
    "PASS: LBXGH decimal structure validated."
)


# ============================================================
# 9. CDC LBXGH RANGE / COUNT CHECK
# ============================================================

#
# CDC reports:
#
#     Glycohemoglobin:
#         3.8 to 17
#
#     Count:
#         6,326
#
#     Missing:
#         418
#
#
# We preserve the actual XPT numeric value 17.0.
#

lbxgh_range_count = int(
    df["LBXGH"]
    .between(
        3.8,
        17.0
    )
    .sum()
)


print(
    "\n--- CDC LBXGH CHECK ---"
)

print(
    "3.8-17.0 count:",
    f"{lbxgh_range_count:,}"
)


if lbxgh_range_count != 6326:

    raise ValueError(
        "LBXGH CDC range count differs."
    )


if int(
    df["LBXGH"]
    .isna()
    .sum()
) != 418:

    raise ValueError(
        "LBXGH CDC missing count differs."
    )


print(
    "PASS: LBXGH matches CDC range/count."
)


# ============================================================
# 10. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 83732:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 93702:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN found."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 83732-93702 validated."
)


# ============================================================
# 11. CONFIRM LBXGH REMAINS FLOATING POINT
# ============================================================

#
# We intentionally do NOT alter LBXGH.
#

if not pd.api.types.is_float_dtype(
    df["LBXGH"].dtype
):

    raise ValueError(
        "LBXGH is no longer floating point."
    )


print(
    "\nPASS: LBXGH remains floating-point."
)


# ============================================================
# 12. CONFIRM NO ZEROS CREATED
# ============================================================

#
# Original GHB_I contained:
#
#     tiny values      = 0
#     ordinary zeros   = 0
#
# Therefore cleaning must not create zero values.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO CHECK ---"
)

print(
    final_zero_counts
)


if final_zero_counts:

    raise ValueError(
        "Unexpected zero values were created:\n"
        + str(final_zero_counts)
    )


print(
    "PASS: No zero values created."
)


# ============================================================
# 13. NUMERIC FIDELITY BEFORE EXPORT
# ============================================================

#
# No numeric correction was necessary.
#
# Therefore:
#
#     SEQN
#     LBXGH
#
# must remain numerically identical to source XPT.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(pre_export_failures)
    )


print(
    "PASS: Both variables preserve "
    "source XPT numeric values exactly."
)


# ============================================================
# 14. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# DO NOT:
#
#     df.round(...)
#
# DO NOT:
#
#     float_format=
#
# DO NOT:
#
#     convert LBXGH to Int64
#
#
# Result:
#
# SEQN
#     83732.0 -> 83732
#
# LBXGH
#     5.7  -> 5.7
#     6.0  -> 6.0
#     17.0 -> 17.0
#
#
# Genuine lab decimal remains AS IS.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 15. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column names/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 16. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT XPT -> CSV NUMERIC FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: 0 numeric differences across "
    "both GHB_I variables."
)


# ============================================================
# 17. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 18. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 19. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found in CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 20. SEQN .0 CHECK
# ============================================================

#
# IMPORTANT:
#
# Only SEQN is an integer field.
#
# DO NOT perform this test on LBXGH.
#

seqn_dot_zero_count = int(
    (
        raw_csv_df["SEQN"]
        .ne("")
        &
        raw_csv_df["SEQN"]
        .str.endswith(".0")
    )
    .sum()
)


print(
    "\n--- SEQN .0 CHECK ---"
)

print(
    "SEQN '.0' values:",
    seqn_dot_zero_count
)


if seqn_dot_zero_count != 0:

    raise ValueError(
        "SEQN still contains '.0'."
    )


print(
    "PASS: SEQN contains no unnecessary '.0'."
)


# ============================================================
# 21. LEGITIMATE LBXGH .0 VALUES
# ============================================================

#
# INFORMATIONAL + FIDELITY CHECK.
#
# LBXGH is continuous.
#
# Whole-looking values such as:
#
#     5.0
#     6.0
#     7.0
#     17.0
#
# are legitimate HbA1c results.
#
# They should remain decimal representations.
#

lbxgh_dot_zero_count = int(
    (
        raw_csv_df["LBXGH"]
        .ne("")
        &
        raw_csv_df["LBXGH"]
        .str.endswith(".0")
    )
    .sum()
)


print(
    "\n--- LEGITIMATE LBXGH .0 VALUES ---"
)

print(
    "LBXGH '.0' values:",
    f"{lbxgh_dot_zero_count:,}"
)


if lbxgh_dot_zero_count != 612:

    raise ValueError(
        "Unexpected number of whole-looking "
        "LBXGH decimal values."
    )


print(
    "PASS: 612 legitimate LBXGH '.0' values "
    "were preserved."
)


# ============================================================
# 22. EXACT LBXGH DECIMAL FIDELITY
# ============================================================

#
# This is the most important HbA1c fidelity check.
#
# rtol = 0
# atol = 0
#
# No numerical difference is accepted.
#

expected_lbxgh = (
    expected_df["LBXGH"]
    .to_numpy(
        dtype=float
    )
)


exported_lbxgh = (
    roundtrip_df["LBXGH"]
    .to_numpy(
        dtype=float
    )
)


lbxgh_exact = np.allclose(
    expected_lbxgh,
    exported_lbxgh,
    rtol=0,
    atol=0,
    equal_nan=True
)


if not lbxgh_exact:

    raise ValueError(
        "LBXGH decimal fidelity failed."
    )


print(
    "\nPASS: Every LBXGH value preserved exactly."
)


# ============================================================
# 23. FINAL LBXGH RANGE CHECK
# ============================================================

final_lbxgh = pd.to_numeric(
    raw_csv_df["LBXGH"]
    .where(
        raw_csv_df["LBXGH"] != "",
        np.nan
    ),
    errors="coerce"
)


final_lbxgh_nonmissing = int(
    final_lbxgh
    .notna()
    .sum()
)


final_lbxgh_missing = int(
    final_lbxgh
    .isna()
    .sum()
)


final_lbxgh_min = float(
    final_lbxgh.min()
)


final_lbxgh_max = float(
    final_lbxgh.max()
)


print(
    "\n--- FINAL LBXGH RANGE ---"
)

print(
    "Nonmissing:",
    f"{final_lbxgh_nonmissing:,}"
)

print(
    "Missing:",
    f"{final_lbxgh_missing:,}"
)

print(
    "Minimum:",
    repr(final_lbxgh_min)
)

print(
    "Maximum:",
    repr(final_lbxgh_max)
)


if final_lbxgh_nonmissing != 6326:

    raise ValueError(
        "Final LBXGH nonmissing count changed."
    )


if final_lbxgh_missing != 418:

    raise ValueError(
        "Final LBXGH missing count changed."
    )


if final_lbxgh_min != 3.8:

    raise ValueError(
        "Final LBXGH minimum changed."
    )


if final_lbxgh_max != 17.0:

    raise ValueError(
        "Final LBXGH maximum changed."
    )


print(
    "PASS: Final LBXGH range/count preserved."
)


# ============================================================
# 24. FINAL ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO CHECK ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts:

    raise ValueError(
        "Unexpected zero values in final CSV:\n"
        + str(csv_zero_counts)
    )


print(
    "PASS: Final CSV contains no zero values."
)


# ============================================================
# 25. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    scientific_count += int(
        raw_csv_df[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 26. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 83732:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 93702:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN in final CSV."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 27. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "GHB_I CONVERSION COMPLETE"
)

print(
    "=" * 100
)


print(
    "Cycle: 2015-2016"
)


print(
    "Component: Laboratory - Glycohemoglobin"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Original ordinary zeros:",
    true_zero_total
)


print(
    "Validated zero restorations:",
    "NONE"
)


print(
    "Whole-number/ID variables:",
    INTEGER_COLUMNS
)


print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)


print(
    "LBXGH nonmissing:",
    f"{lbxgh_nonmissing:,}"
)


print(
    "LBXGH missing:",
    f"{lbxgh_missing:,}"
)


print(
    "LBXGH range:",
    "3.8-17.0 %"
)


print(
    "LBXGH fractional observations:",
    f"{lbxgh_fractional:,}"
)


print(
    "LBXGH whole-looking observations:",
    f"{lbxgh_whole_looking:,}"
)


print(
    "Legitimate LBXGH '.0' CSV values:",
    f"{lbxgh_dot_zero_count:,}"
)


print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)


print(
    "Missing-position differences:",
    "0 expected"
)


print(
    "LBXGH decimal differences:",
    "0 expected"
)


print(
    "Tiny values remaining:",
    "0 expected"
)


print(
    "SEQN '.0' values:",
    "0 expected"
)


print(
    "LBXGH '.0' values:",
    "ALLOWED / PRESERVED"
)


print(
    "Scientific notation:",
    "0 expected"
)


print(
    "\nFINAL RULE:"
)


print(
    "LBXGH remains a genuine decimal laboratory value."
)


print(
    "No HbA1c value was rounded or rewritten."
)

NHANES 2015-2016 GHB_I XPT -> CSV
Rows: 6,744
Columns: 2
PASS: Structure = 6,744 rows x 2 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: ['LBXGH']
Whole-number/ID fields: ['SEQN']

--- ORIGINAL XPT ZERO CHECK ---
Tiny XPORT artifacts: 0
Ordinary numeric zeros: 0
PASS: GHB_I contains 0 tiny artifacts and 0 ordinary numeric zeros.

PASS: No tiny-to-zero correction required.

--- GENUINE DECIMAL DETECTION ---
SEQN: 0 fractional observations
LBXGH: 5,714 fractional observations

PASS: LBXGH is the only genuine decimal field.

--- LBXGH DECIMAL DETAILS ---
Nonmissing: 6,326
Missing: 418
Minimum: 3.8
Maximum: 17.0
Fractional observations: 5,714
Whole-looking observations: 612
PASS: LBXGH decimal structure validated.

--- CDC LBXGH CHECK ---
3.8-17.0 count: 6,326
PASS: LBXGH matches CDC range/count.

--- SEQN ---
Minimum: 83732
Maximum: 93702
Missing: 0
Duplicates: 0
PASS: SEQN 83732-93702 validated.

PASS: LBXGH remains floating-point.

--- FINAL ZERO CHECK ---
{}
PASS: No zero va